[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Qwen35_0p8B_All24_WindowAnchor_V3_Optimized_Colab.ipynb)

# Qwen3.5-0.8B | All-24 Window+Anchor V3 — Quality & Kernel Optimization

## What the V2 statistics showed
- DEV: **59.38% teacher agreement**, mean JS **0.2319**, NLL ratio **1.362×**.
- Sealed HOLDOUT: **59.38% teacher agreement** (57/96), mean JS **0.2004**. Gold-token perplexity ratio **0.725×**, but this is NOT evidence of fidelity because teacher agreement remains low.
- FP16 no-cache prefill speedup: **1.036× / 1.044× / 0.941×** at 64/128/192 tokens.
- All five student chat outputs collapsed to repetitive `. . .`; therefore **no Hugging Face upload** was appropriate.
- Layerwise native-teacher MSE decreased in all 18 Delta substitutions, yet model output did not recover; small local MSE is insufficient when errors compound across 24 residual blocks.

## V3 changes driven by that evidence
1. **Faster attention:** one SDPA call rather than multiple tiny query chunks for contexts up to 512 tokens; full GQA uses one native-equivalent causal SDPA call. Long contexts retain window/anchor chunks.
2. **More expressive nonrecurrent Delta substitution:** per-output affine calibration (trainable gain/bias) on top of V2 learned recency, temperature, and direct-value route. No CeNN or native recurrent DeltaNet in student.
3. **Better training:** 64 train-derived examples per layer, 100 local steps; 700 global KD steps, rank-24 adapters and stronger checkpoint selection. FFNs and decoder norms frozen.
4. **Real independent splits:** WikiText TRAIN/VALIDATION/TEST rather than slicing one training split. Additional **teacher-generated chat continuation prefixes** mixed into training to address punctuation-collapse on actual chat inputs. Evaluation chats are distinct.
5. **Honest qualification:** strict independent DEV and sealed HOLDOUT fidelity, perplexity/JS, nonrepetitive chat outputs, model reload, and end-to-end speed. Unqualified models are never uploaded to HF.
6. **Artifacts:** always save final statistics ZIP for download; standalone FP16 model ZIP also available even if quality gate fails (unless export itself fails).

**Important:** This is still a text-only **no-KV-cache** research substitution. DeltaNet may inherently outperform local attention on longer sequences. No improvement is claimed until the new experiment runs.

**Verification:** the fused 64–192-token attention path was numerically cross-checked against its chunked reference; this does not replace a full GPU Qwen run. The updated tokenizer fallback and non-repetition gate are intended to prevent V2 chat-test regressions.


In [1]:
#@title 1. Setup (Qwen3.5-compatible Transformers)
%pip -q install -U "git+https://github.com/huggingface/transformers.git@main" "accelerate>=1.0" "datasets>=3" "huggingface_hub>=0.34" "safetensors>=0.5"
%pip -q install --no-deps "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@66a244e8e8ee5c8b9b44f15a888d75e59761e561"

import os, json, math, random, time, gc, hashlib, copy, shutil, zipfile, traceback
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from datasets import load_dataset
from huggingface_hub import HfApi, login, get_token
from safetensors.torch import save_model
from tinycenn_lm import qwen35_window_anchor_v3 as wa

SEED=20261018
MODEL_ID="Qwen/Qwen3.5-0.8B"
DELTA_WINDOW=128 #@param {type:"integer"}
FULL_WINDOW=512 #@param {type:"integer"}
ANCHORS=16 #@param {type:"integer"}
CHUNK=64 #@param {type:"integer"}
MAX_TOKENS=192 #@param {type:"integer"}
TRAIN_EXAMPLES=480 #@param {type:"integer"}
DEV_EXAMPLES=96 #@param {type:"integer"}
HOLDOUT_EXAMPLES=128 #@param {type:"integer"}
TRAIN_STEPS=700 #@param {type:"integer"}
GRAD_ACCUM=2 #@param {type:"integer"}
LORA_RANK=24 #@param {type:"integer"}
LR_START=0.0005 #@param {type:"number"}
LR_END=0.000025 #@param {type:"number"}
EVAL_EVERY=50
EVAL_PATIENCE=6  # Stop if six consecutive DEV checks do not improve
TRAIN_CHAT_PROMPTS=40 #@param {type:"integer"}
CHAT_TEACHER_STEPS=12 #@param {type:"integer"}
CHAT_TEACHER_STRIDE=3 #@param {type:"integer"}
LOCAL_EXAMPLES=64 #@param {type:"integer"}
LOCAL_VAL=4
LOCAL_STEPS=100 #@param {type:"integer"}
MAX_GRAD_NORM=0.6
DISTILL_TEMP=2.0
GOLD_CE_WEIGHT=0.05
AUTO_DOWNLOAD_STATS=True #@param {type:"boolean"}
AUTO_DOWNLOAD_MODEL_ZIP=False #@param {type:"boolean"}
UPLOAD_IF_QUALIFIED=True #@param {type:"boolean"}
HF_PRIVATE=True #@param {type:"boolean"}
HF_NAME="qwen35-0p8b-window-anchor-v3-optimized" #@param {type:"string"}
MIN_TEACHER_AGREEMENT=0.90
MAX_PERPLEXITY_RATIO=1.25
MAX_JS=0.045
MIN_SPEEDUP=1.00

BASE=Path("/content/Qwen35_0p8B_All24_WindowAnchor_V3")
ARTIFACTS=BASE/"exports"
REPORTS=BASE/"results"
ARTIFACTS.mkdir(parents=True,exist_ok=True)
REPORTS.mkdir(parents=True,exist_ok=True)
assert DELTA_WINDOW>=16 and FULL_WINDOW>=MAX_TOKENS and 0<=ANCHORS<=DELTA_WINDOW and CHUNK>=16
assert LOCAL_VAL>=2 and LOCAL_EXAMPLES<=TRAIN_EXAMPLES
assert TRAIN_CHAT_PROMPTS>0 and CHAT_TEACHER_STEPS>0 and CHAT_TEACHER_STRIDE>0
assert 0<LR_END<LR_START and TRAIN_STEPS>0 and GRAD_ACCUM>0
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
assert torch.cuda.is_available(),"Enable GPU in Runtime > Change runtime type"
device=torch.device("cuda",torch.cuda.current_device())
torch.cuda.manual_seed_all(SEED)
print("GPU:",torch.cuda.get_device_name(),"source",MODEL_ID)
print("Experiment: replace 18 DeltaNet AND 6 full attention. No CeNN.")
# Cheap CPU causality smoke test before downloading the 1.75GB teacher weights.
a=torch.randn(1,2,10,8)
b=torch.randn(1,2,10,8)
v=torch.randn(1,2,10,8)
ref=wa.causal_window_anchor(a,b,v,None,window=3,anchors=2,chunk_size=4)
v[:,:,8:]+=1000
out=wa.causal_window_anchor(a,b,v,None,window=3,anchors=2,chunk_size=4)
torch.testing.assert_close(ref[:,:,:8],out[:,:,:8],atol=1e-5,rtol=1e-5)
print("PASS: causal window/anchor mixer has no future-token leakage.")
# Mathematical parity: optimized single-call masked SDPA and old chunked
# attention must agree on the same 128-token sequence before GPU training.
q=torch.randn(1,4,128,16)
k=torch.randn(1,4,128,16)
v=torch.randn(1,4,128,16)
scale=torch.tensor([6.,7.,5.,9.])
decay=torch.tensor([.01,.02,.005,.03])
optimized=wa.causal_window_anchor_biased(
    q,k,v,None,64,8,32,scale,decay,dense_threshold=512)
reference=wa.causal_window_anchor_biased(
    q,k,v,None,64,8,32,scale,decay,dense_threshold=0)
torch.testing.assert_close(optimized,reference,atol=1e-5,rtol=1e-4)
print("PASS: fused/long-form chunked attention numerical parity.")



  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 18.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
GPU: Tesla T4 source Qwen/Qwen3.5-0.8B
Experiment: replace 18 DeltaNet AND 6 full attention. No CeNN.
PASS: causal window/anchor mixer has no future-token leakage.
PASS: fused/long-form chunked attention numerical parity.


In [2]:
#@title 2. Three official WikiText splits: TRAIN, DEV, sealed HOLDOUT
# Unlike V2, validation/test are not slices of the training split.
DATA_SOURCE="wikitext-2-raw-v1"
def clean_texts(split,needed):
    ds=load_dataset("Salesforce/wikitext","wikitext-2-raw-v1",split=split)
    unique=[];seen=set()
    for row in ds:
        value=" ".join(str(row["text"]).split())
        if 80<=len(value)<=850 and not value.startswith("=") and value not in seen:
            unique.append(value);seen.add(value)
    if len(unique)<needed:
        raise RuntimeError(f"Split {split} has only {len(unique)} eligible rows; need {needed}")
    rng=random.Random(SEED+{"train":11,"validation":23,"test":41}[split])
    rng.shuffle(unique)
    return unique[:needed]
try:
    train_text=clean_texts("train",TRAIN_EXAMPLES)
    dev_text=clean_texts("validation",DEV_EXAMPLES)
    holdout_text=clean_texts("test",HOLDOUT_EXAMPLES)
except Exception as exc:
    print("Official data unavailable; deterministic fallback for debugging ONLY:",repr(exc))
    DATA_SOURCE="synthetic_fallback_not_valid_for_publication"
    subjects=["A scientist","A library","An engineer","A hospital","A teacher",
              "A spacecraft","A mathematician","An orchestra","A historian","A botanist"]
    verbs=["described","studied","measured","compared","discussed",
           "reconsidered","observed","evaluated","investigated"]
    suffixes=["a new method using controlled experiments and careful analysis.",
              "the importance of verification and repeating the measurement.",
              "several observations that challenged the previous explanation.",
              "the distinction between efficiency, accuracy and usefulness.",
              "how practical design choices affected the final result."]
    needed=TRAIN_EXAMPLES+DEV_EXAMPLES+HOLDOUT_EXAMPLES
    unique=[(f"{subjects[i%len(subjects)]} "
             f"{verbs[(i//len(subjects))%len(verbs)]} "
             f"{suffixes[(i//(len(subjects)*len(verbs)))%len(suffixes)]} "
             f"Deterministic synthetic debug record {i}.")
            for i in range(needed)]
    print("WARNING: synthetic data is NOT a quality benchmark and HF upload will be blocked.")
    random.Random(SEED).shuffle(unique)
    train_text=unique[:TRAIN_EXAMPLES]
    dev_text=unique[TRAIN_EXAMPLES:TRAIN_EXAMPLES+DEV_EXAMPLES]
    holdout_text=unique[TRAIN_EXAMPLES+DEV_EXAMPLES:
                        TRAIN_EXAMPLES+DEV_EXAMPLES+HOLDOUT_EXAMPLES]
assert not set(train_text)&set(dev_text)
assert not set(train_text)&set(holdout_text)
assert not set(dev_text)&set(holdout_text)
print(DATA_SOURCE,"official split sizes",len(train_text),len(dev_text),len(holdout_text))

# Separate teacher-guided chat continuation samples will be constructed in
# Cell 3, exclusively for TRAIN. Do not contaminate DEV or HOLDOUT.
CHAT_TOPICS=[
  "weather patterns","photovoltaic panels","rail transport","public libraries",
  "flight safety","quantum computing","solar eclipses","the water cycle",
  "linear algebra","Newton's laws","open-source software","urban gardening",
  "distributed databases","electric motors","sound waves","sustainable housing",
  "marine biology","computer networks","public health","the history of printing",
  "classical music","the Fourier transform","recycling plastics","astronomy",
  "mountain ecosystems","learning a language","navigation","digital privacy",
  "mathematical proof","robotic control","operating systems","power grids",
  "geography","meteor showers","cell biology","materials science",
  "renewable energy","an optical telescope","machine learning","building bridges",
  "probability","geometric shapes","battery storage","space exploration"
]
CHAT_TEMPLATE_PATTERNS=[
  "Explain {} in two concise sentences.",
  "Describe an everyday example of {}.",
  "What are two important facts about {}?",
  "Write a clear beginner-level explanation of {}.",
]
CHAT_TRAIN_PROMPTS=[
  CHAT_TEMPLATE_PATTERNS[i%len(CHAT_TEMPLATE_PATTERNS)].format(
      CHAT_TOPICS[i%len(CHAT_TOPICS)])
  for i in range(TRAIN_CHAT_PROMPTS)
]
assert len(set(CHAT_TRAIN_PROMPTS))==len(CHAT_TRAIN_PROMPTS)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

wikitext-2-raw-v1 official split sizes 480 96 128


In [3]:
#@title 3. Official Qwen3.5 text-only teacher; cache train and DEV logits once
teacher,tokenizer,source_root=wa.load_official_text(MODEL_ID,dtype=torch.float16)
teacher.to(device).eval().requires_grad_(False)
original_config=teacher.config.to_dict()

def make_record(text):
    ids=tokenizer(text,add_special_tokens=False)["input_ids"][:MAX_TOKENS+1]
    if len(ids)<8:raise ValueError("Text too short")
    return {"input_ids":torch.tensor([ids[:-1]],dtype=torch.long),
            "gold_next":int(ids[-1]),"text":text,"length":len(ids)-1}
train_rows=[make_record(s) for s in train_text]
dev_rows=[make_record(s) for s in dev_text]
# Holdout records are NOT yet materialized or evaluated.
@torch.inference_mode()
def attach_reference(records):
    for index,rec in enumerate(records):
        x=rec["input_ids"].to(device)
        logits=teacher(input_ids=x,use_cache=False,logits_to_keep=1).logits[:, -1]
        rec["teacher_logits"]=logits.float().cpu().contiguous()
        if index%25==0:print("Reference logits",index,"/",len(records))
    return records
train_rows=attach_reference(train_rows)
dev_rows=attach_reference(dev_rows)
# Generate teacher-forced CHAT continuations on TRAIN prompts only.
# V2 trained on prose-only next tokens; at inference the student repeated
# dots on all five chat prompts. This exposes the student to the *same
# chat-template prefix distribution* and multiple continuation positions.
def encode_chat_prompt(prompt):
    msgs=[{"role":"user","content":prompt}]
    try:
        try:
            obj=tokenizer.apply_chat_template(
                msgs,tokenize=True,add_generation_prompt=True,
                return_tensors="pt",enable_thinking=False)
        except TypeError:
            obj=tokenizer.apply_chat_template(
                msgs,tokenize=True,add_generation_prompt=True,return_tensors="pt")
    except Exception as exc:
        print("Chat template unavailable; using text prefix:",type(exc).__name__)
        obj=tokenizer("User: "+prompt+"\nAssistant:",return_tensors="pt")["input_ids"]
    if isinstance(obj,dict):obj=obj["input_ids"]
    if hasattr(obj,"input_ids"):obj=obj.input_ids
    if not isinstance(obj,torch.Tensor):obj=torch.as_tensor(obj,dtype=torch.long)
    if obj.ndim==1:obj=obj[None]
    assert obj.ndim==2
    return obj[:,-MAX_TOKENS:].to(device)
chat_train_records=[]
for j,prompt in enumerate(CHAT_TRAIN_PROMPTS):
    prefix=encode_chat_prompt(prompt)
    for step in range(CHAT_TEACHER_STEPS):
        with torch.inference_mode(),torch.autocast("cuda",dtype=torch.float16):
            predicted=teacher(input_ids=prefix,use_cache=False,
                              logits_to_keep=1).logits[:,-1,:].float()
        token=int(predicted.argmax(-1).item())
        if step%CHAT_TEACHER_STRIDE==0:
            chat_train_records.append({
                "input_ids":prefix.detach().cpu().clone(),
                "teacher_logits":predicted.detach().cpu().contiguous(),
                "gold_next":token,
                "text":prompt,
                "source":"teacher_chat_continuation",
                "length":prefix.shape[-1]})
        next_token=torch.tensor([[token]],device=device,dtype=torch.long)
        prefix=torch.cat((prefix,next_token),dim=-1)[:,-MAX_TOKENS:]
        if token==tokenizer.eos_token_id:break
    if (j+1)%10==0:
        print("Cached training chat continuations:",j+1,"/",len(CHAT_TRAIN_PROMPTS))
assert len(chat_train_records)>=TRAIN_CHAT_PROMPTS
train_rows.extend(chat_train_records)
print("TRAIN cache includes",len(chat_train_records),"chat continuation records; ",
      "total TRAIN",len(train_rows))
teacher.cpu()
gc.collect();torch.cuda.empty_cache()
print("Teacher logits cached, teacher on CPU.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Reference logits 0 / 480
Reference logits 25 / 480
Reference logits 50 / 480
Reference logits 75 / 480
Reference logits 100 / 480
Reference logits 125 / 480
Reference logits 150 / 480
Reference logits 175 / 480
Reference logits 200 / 480
Reference logits 225 / 480
Reference logits 250 / 480
Reference logits 275 / 480
Reference logits 300 / 480
Reference logits 325 / 480
Reference logits 350 / 480
Reference logits 375 / 480
Reference logits 400 / 480
Reference logits 425 / 480
Reference logits 450 / 480
Reference logits 475 / 480
Reference logits 0 / 96
Reference logits 25 / 96
Reference logits 50 / 96
Reference logits 75 / 96
Cached training chat continuations: 10 / 40
Cached training chat continuations: 20 / 40
Cached training chat continuations: 30 / 40
Cached training chat continuations: 40 / 40
TRAIN cache includes 160 chat continuation records;  total TRAIN 640
Teacher logits cached, teacher on CPU.


In [4]:
#@title 4. Replace ALL 24 token mixers; freeze original FFNs and layer norms
def ff_norm_fingerprint(model):
    sha=hashlib.sha256()
    for i,layer in enumerate(model.model.layers):
        for name in ("mlp","input_layernorm","post_attention_layernorm"):
            for k,x in getattr(layer,name).state_dict().items():
                sha.update(f"{i}/{name}/{k}".encode())
                sha.update(x.detach().cpu().contiguous().numpy().tobytes())
    return sha.hexdigest()

TEACHER_FFN_SHA=ff_norm_fingerprint(teacher)
student=copy.deepcopy(teacher)
original_student_layers=tuple(layer.block_type for layer in student.model.layers)
counts=wa.install_all24_v2(student,delta_window=DELTA_WINDOW,full_window=FULL_WINDOW,anchors=ANCHORS,chunk_size=CHUNK)
student.to(device).eval().requires_grad_(False)
assert counts=={"delta_replaced":18,"full_replaced":6}
assert ff_norm_fingerprint(student)==TEACHER_FFN_SHA
assert not any("cenn" in name.lower() for name,_ in student.named_modules())
assert all(not p.requires_grad for p in student.parameters())
print("ALL 24 REPLACED:",counts,"original FFNs + norms exact SHA:",TEACHER_FFN_SHA)

@torch.inference_mode()
def metrics(model,rows):
    model.eval()
    n=0; agree=0; js=0.; kl=0.; student_ce=0.;teacher_ce=0.
    for rec in rows:
        x=rec["input_ids"].to(device)
        with torch.autocast("cuda",dtype=torch.float16):
            logits=model(input_ids=x,use_cache=False,logits_to_keep=1).logits[:, -1]
        a=rec["teacher_logits"].to(device).float()
        b=logits.float()
        pa=a.softmax(-1);pb=b.softmax(-1)
        midpoint=0.5*(pa+pb)
        js+=float(0.5*((pa*(pa.clamp_min(1e-9).log()-midpoint.clamp_min(1e-9).log())).sum()
                        +(pb*(pb.clamp_min(1e-9).log()-midpoint.clamp_min(1e-9).log())).sum()))
        kl+=float(F.kl_div(b.log_softmax(-1),pa,reduction="sum"))
        agree+=int(a.argmax(-1).item()==b.argmax(-1).item())
        goal=torch.tensor([rec["gold_next"]],dtype=torch.long,device=device)
        student_ce+=float(F.cross_entropy(b,goal))
        teacher_ce+=float(F.cross_entropy(a,goal))
        n+=1
    return {"n":n,"top1_agreement":agree/n,"mean_js":js/n,
            "mean_kl":kl/n,"student_nll":student_ce/n,
            "teacher_nll":teacher_ce/n,
            "ppl_ratio":math.exp(min(30,(student_ce-teacher_ce)/n))}
BASELINE_DEV=metrics(student,dev_rows)
print("Untuned all-24 substitute DEV:",BASELINE_DEV)
(REPORTS/"architecture.json").write_text(json.dumps({
    "source":MODEL_ID,"source_layout":original_student_layers,
    "replaced":counts,"no_cenn":True,"no_delta_recurrence":True,
    "delta_window":DELTA_WINDOW,"full_window":FULL_WINDOW,"anchors":ANCHORS,"chunk_size":CHUNK,
    "ffn_norm_sha256":TEACHER_FFN_SHA,"train_data":DATA_SOURCE
},indent=2))


ALL 24 REPLACED: {'delta_replaced': 18, 'full_replaced': 6} original FFNs + norms exact SHA: 9aa9401bf55838aae15daedd6726accfbc0478315952814fbd08f63775ebc2d5
Untuned all-24 substitute DEV: {'n': 96, 'top1_agreement': 0.0, 'mean_js': 0.6838204879313707, 'mean_kl': 12.519155492385229, 'student_nll': 16.087554703156155, 'teacher_nll': 1.2716398036961134, 'ppl_ratio': 2719381.164838158}


952

In [5]:
#@title 5. Stage A: 18-layer teacher-activation alignment on TRAIN data only
from collections import defaultdict
delta_ids=[i for i,L in enumerate(teacher.model.layers)
           if L.block_type=="linear_attention"]
assert len(delta_ids)==18
teacher.to(device).eval()
cache=defaultdict(list)
inbuf={};outbuf={};hooks=[]
def input_hook(i):
    def call(m,args,y):inbuf[i]=y.detach().to("cpu",dtype=torch.float16)
    return call
def output_hook(i):
    def call(m,args,y):
        if isinstance(y,tuple):y=y[0]
        outbuf[i]=y.detach().to("cpu",dtype=torch.float16)
    return call
for i in delta_ids:
    L=teacher.model.layers[i]
    hooks.append(L.input_layernorm.register_forward_hook(input_hook(i)))
    hooks.append(L.linear_attn.register_forward_hook(output_hook(i)))
try:
    for j,record in enumerate(train_rows[:LOCAL_EXAMPLES]):
        inbuf.clear();outbuf.clear()
        with torch.inference_mode(),torch.autocast("cuda",dtype=torch.float16):
            teacher(input_ids=record["input_ids"].to(device),
                    use_cache=False,logits_to_keep=1)
        assert set(inbuf)==set(delta_ids) and set(outbuf)==set(delta_ids)
        for i in delta_ids:cache[i].append((inbuf[i],outbuf[i]))
        if (j+1)%8==0:print("Native teacher activations:",j+1)
finally:
    for h in hooks:h.remove()
    teacher.cpu();gc.collect();torch.cuda.empty_cache()
class Adapter(nn.Module):
    def __init__(self,base):
        super().__init__()
        self.base=base.requires_grad_(False)
        self.A=nn.Parameter(torch.empty(LORA_RANK,base.in_features,
                             device=device,dtype=torch.float32))
        self.B=nn.Parameter(torch.zeros(base.out_features,LORA_RANK,
                             device=device,dtype=torch.float32))
        nn.init.normal_(self.A,std=0.01)
        self.scale=1/LORA_RANK
    def forward(self,x):
        return self.base(x)+F.linear(F.linear(x,self.A),self.B)*self.scale
    @torch.no_grad()
    def merge(self):
        self.base.weight.add_(((self.B@self.A)*self.scale).to(self.base.weight.dtype))
        return self.base
@torch.no_grad()
def mixer_error(m,pairs):
    values=[]
    for inp,target in pairs:
        x=inp.to(device);y=target.to(device)
        with torch.autocast("cuda",dtype=torch.float16):
            pred=m(hidden_states=x,attention_mask=torch.ones(
                x.shape[:2],device=device,dtype=torch.long),cache_params=None)
        values.append(float(F.mse_loss(pred.float(),y.float())))
    return float(np.mean(values))
LOCAL_HISTORY=[]
for count,i in enumerate(delta_ids,1):
    mixer=student.model.layers[i].linear_attn
    assert isinstance(mixer,wa.WindowAnchorDeltaV2)
    mixer.requires_grad_(False)
    wrappers={}
    pars=[]
    for name in ("in_proj_qkv","in_proj_b","in_proj_z","out_proj"):
        wrapper=Adapter(getattr(mixer,name))
        setattr(mixer,name,wrapper)
        wrappers[name]=wrapper;pars.extend([wrapper.A,wrapper.B])
    for p in (mixer.log_temperature,mixer.log_decay,mixer.direct_mix_logit,
              mixer.output_gain,mixer.output_bias):
        p.requires_grad_(True);pars.append(p)
    # Internal Delta mixer normalization can adapt to the changed core.
    # Decoder block norms and all FFNs remain frozen and fingerprinted.
    # GradScaler cannot unscale FP16 parameter gradients. Store the
    # trainable internal Qwen Delta-gate norm in FP32 while retaining FP16
    # autocast computation; convert it back on final model.half().
    mixer.norm.float().requires_grad_(True)
    pars.extend([p for p in mixer.norm.parameters() if p.requires_grad])
    assert all(p.dtype==torch.float32 for p in pars)
    examples=cache[i]
    train=examples[:-LOCAL_VAL];val=examples[-LOCAL_VAL:]
    initial=mixer_error(mixer,val);best=initial;beststep=0
    snap=[p.detach().cpu().clone() for p in pars]
    opt=torch.optim.AdamW(pars,lr=4e-4)
    scaler_local=torch.amp.GradScaler("cuda")
    rng=random.Random(SEED+i)
    for step in range(1,LOCAL_STEPS+1):
        x,y=train[rng.randrange(len(train))]
        x=x.to(device);y=y.to(device)
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda",dtype=torch.float16):
            pred=mixer(hidden_states=x,attention_mask=torch.ones(
                x.shape[:2],device=device,dtype=torch.long),cache_params=None)
            loss=F.smooth_l1_loss(pred.float(),y.float(),beta=0.2)
            loss=loss+0.05*(1-F.cosine_similarity(
                pred.float(),y.float(),dim=-1).mean())
        if not bool(torch.isfinite(loss).item()):
            raise FloatingPointError(f"Layer {i}: loss nonfinite")
        scaler_local.scale(loss).backward()
        scaler_local.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(pars,0.7)
        scaler_local.step(opt);scaler_local.update()
        if step%10==0 or step==LOCAL_STEPS:
            err=mixer_error(mixer,val)
            if err<best:
                best=err;beststep=step
                snap=[p.detach().cpu().clone() for p in pars]
    with torch.no_grad():
        for p,old in zip(pars,snap):p.copy_(old.to(p.device))
    for name,w in wrappers.items():setattr(mixer,name,w.merge())
    after=mixer_error(mixer,val)
    LOCAL_HISTORY.append({"layer":i,"initial_mse":initial,
                          "final_mse":after,"selected_step":beststep})
    print(f"Stage A Delta {count}/18 L{i}: {initial:.5f} → {after:.5f}")
    del opt,scaler_local
    gc.collect()
(REPORTS/"local_teacher_alignment.json").write_text(json.dumps({
    "train_examples":LOCAL_EXAMPLES-LOCAL_VAL,
    "validation_examples":LOCAL_VAL,"steps_per_layer":LOCAL_STEPS,
    "results":LOCAL_HISTORY},indent=2))
del cache,inbuf,outbuf
gc.collect();torch.cuda.empty_cache()
assert ff_norm_fingerprint(student)==TEACHER_FFN_SHA
LOCAL_DEV=metrics(student,dev_rows)
(REPORTS/"local_teacher_alignment.json").write_text(json.dumps({
    "train_examples":LOCAL_EXAMPLES-LOCAL_VAL,
    "validation_examples":LOCAL_VAL,"steps_per_layer":LOCAL_STEPS,
    "results":LOCAL_HISTORY,
    "untuned_dev":BASELINE_DEV,"post_local_dev":LOCAL_DEV},indent=2))
print("DEV after 18 layerwise alignments:",LOCAL_DEV)
print("All 18 Delta substitutions locally trained; starting joint KD.")


Native teacher activations: 8
Native teacher activations: 16
Native teacher activations: 24
Native teacher activations: 32
Native teacher activations: 40
Native teacher activations: 48
Native teacher activations: 56
Native teacher activations: 64
Stage A Delta 1/18 L0: 0.00870 → 0.00412
Stage A Delta 2/18 L1: 0.01606 → 0.00632
Stage A Delta 3/18 L2: 0.01606 → 0.00650
Stage A Delta 4/18 L4: 0.00419 → 0.00189
Stage A Delta 5/18 L5: 0.00523 → 0.00210
Stage A Delta 6/18 L6: 0.00508 → 0.00228
Stage A Delta 7/18 L8: 0.00678 → 0.00192
Stage A Delta 8/18 L9: 0.00111 → 0.00070
Stage A Delta 9/18 L10: 0.00847 → 0.00340
Stage A Delta 10/18 L12: 0.00103 → 0.00066
Stage A Delta 11/18 L13: 0.00911 → 0.00245
Stage A Delta 12/18 L14: 0.00727 → 0.00248
Stage A Delta 13/18 L16: 0.00386 → 0.00175
Stage A Delta 14/18 L17: 0.00090 → 0.00049
Stage A Delta 15/18 L18: 0.00101 → 0.00063
Stage A Delta 16/18 L20: 0.00140 → 0.00086
Stage A Delta 17/18 L21: 0.00210 → 0.00093
Stage A Delta 18/18 L22: 0.01359 → 0.00

In [6]:
#@title 5. LoRA distillation of all 24 replacement mixers (FP16 + frozen FFNs)
class LoRA(nn.Module):
    def __init__(self,base,rank=LORA_RANK):
        super().__init__()
        assert isinstance(base,nn.Linear)
        self.base=base.requires_grad_(False)
        d=base.weight.device
        self.A=nn.Parameter(torch.empty(rank,base.in_features,dtype=torch.float32,device=d))
        self.B=nn.Parameter(torch.zeros(base.out_features,rank,dtype=torch.float32,device=d))
        nn.init.normal_(self.A,std=0.01)
        self.scale=1/rank
    def forward(self,x):
        return self.base(x)+F.linear(F.linear(x,self.A),self.B)*self.scale
    @torch.no_grad()
    def merge(self):
        delta=(self.B.float()@self.A.float())*self.scale
        self.base.weight.add_(delta.to(self.base.weight.dtype))
        return self.base

trainable=[]
for layer in student.model.layers:
    module=(layer.linear_attn if layer.block_type=="linear_attention" else layer.self_attn)
    names=("in_proj_qkv","in_proj_b","in_proj_z","out_proj") if isinstance(module,wa.WindowAnchorDeltaV2) else ("q_proj","k_proj","v_proj","o_proj")
    for name in names:
        wrapped=LoRA(getattr(module,name))
        setattr(module,name,wrapped)
        trainable.extend((wrapped.A,wrapped.B))
    if isinstance(module,wa.WindowAnchorDeltaV2):
        trainable.extend((module.log_temperature,module.log_decay,
                          module.direct_mix_logit,module.output_gain,
                          module.output_bias))
        module.norm.float().requires_grad_(True)
        trainable.extend([p for p in module.norm.parameters() if p.requires_grad])
assert all(p.dtype==torch.float32 for p in trainable)
assert all(not p.requires_grad for layer in student.model.layers for p in layer.mlp.parameters())
assert all(not p.requires_grad for layer in student.model.layers
           for p in layer.input_layernorm.parameters())

optimizer=torch.optim.AdamW(trainable,lr=LR_START,weight_decay=0.0)
scaler=torch.amp.GradScaler("cuda")
best=[p.detach().cpu().clone() for p in trainable]
best_step=0
best_eval=LOCAL_DEV
history=[{"step":0,"dev":LOCAL_DEV,"phase":"post_local_teacher_alignment"}]
stale_evals=0
actual_train_steps=0
order=list(range(len(train_rows)))
rng=random.Random(SEED+50)
ix=0
student.eval()
def select_key(x):
    score=x["top1_agreement"]-0.40*max(0,x["ppl_ratio"]-1.0)-0.70*x["mean_js"]
    return (round(score,7),x["top1_agreement"],-x["mean_js"])

for step in range(1,TRAIN_STEPS+1):
    actual_train_steps=step
    eta=(step-1)/max(1,TRAIN_STEPS-1)
    lr=LR_END+0.5*(LR_START-LR_END)*(1+math.cos(math.pi*eta))
    for g in optimizer.param_groups:g["lr"]=lr
    optimizer.zero_grad(set_to_none=True)
    parts=[]
    for _ in range(GRAD_ACCUM):
        if ix==0:rng.shuffle(order)
        rec=train_rows[order[ix]];ix=(ix+1)%len(order)
        tokens=rec["input_ids"].to(device)
        gold=torch.tensor([rec["gold_next"]],device=device)
        teacher_logits=rec["teacher_logits"].to(device).float()
        with torch.autocast("cuda",dtype=torch.float16):
            out=student(input_ids=tokens,use_cache=False,logits_to_keep=1)
            p=out.logits[:, -1].float()
            q=teacher_logits
            tau=DISTILL_TEMP
            kd=F.kl_div(F.log_softmax(p/tau,dim=-1),
                        F.softmax(q/tau,dim=-1),reduction="batchmean")*tau*tau
            ce=F.cross_entropy(p,gold)
            loss=kd+GOLD_CE_WEIGHT*ce
        if not bool(loss.isfinite().item()):
            raise FloatingPointError(f"Invalid loss at step {step}")
        scaler.scale(loss/GRAD_ACCUM).backward()
        parts.append(float(loss.detach()))
    scaler.unscale_(optimizer)
    norm=torch.nn.utils.clip_grad_norm_(trainable,MAX_GRAD_NORM)
    if bool(torch.isfinite(norm).item()):
        scaler.step(optimizer)
    else:print("Nonfinite gradient skipped:",step)
    scaler.update()
    if step%EVAL_EVERY==0 or step==1 or step==TRAIN_STEPS:
        current=metrics(student,dev_rows)
        if select_key(current)>select_key(best_eval):
            best_eval=current
            best_step=step
            stale_evals=0
            best=[p.detach().cpu().clone() for p in trainable]
        else:
            stale_evals+=1
        record={"step":step,"lr":lr,"loss":float(np.mean(parts)),
                "dev":current,"best_step":best_step}
        history.append(record)
        (REPORTS/"training_history.json").write_text(json.dumps(history,indent=2))
        print(f"Step {step}/{TRAIN_STEPS}: teacher match {current['top1_agreement']:.3f}, "
              f"PPL ratio {current['ppl_ratio']:.3f}, best {best_step}")
        if stale_evals>=EVAL_PATIENCE and step>=350:
            print("DEV early stop after",stale_evals,
                  "non-improving evaluations; best step",best_step)
            break
with torch.no_grad():
    for param,snapshot in zip(trainable,best):
        param.copy_(snapshot.to(param.device))
for layer in student.model.layers:
    mod=layer.linear_attn if layer.block_type=="linear_attention" else layer.self_attn
    names=("in_proj_qkv","in_proj_b","in_proj_z","out_proj") if isinstance(mod,wa.WindowAnchorDeltaV2) else ("q_proj","k_proj","v_proj","o_proj")
    for name in names:
        wrapper=getattr(mod,name)
        assert isinstance(wrapper,LoRA)
        setattr(mod,name,wrapper.merge())
del optimizer,best
gc.collect();torch.cuda.empty_cache()
assert ff_norm_fingerprint(student)==TEACHER_FFN_SHA
student.half()  # include trained FP32 recency/temperature gains in FP16 export
FINAL_DEV=metrics(student,dev_rows)
print("Best training step",best_step,"FINAL DEV",FINAL_DEV)


Nonfinite gradient skipped: 1
Step 1/700: teacher match 0.021, PPL ratio 159.902, best 0
Nonfinite gradient skipped: 2
Nonfinite gradient skipped: 3
Nonfinite gradient skipped: 4
Nonfinite gradient skipped: 5
Nonfinite gradient skipped: 6
Nonfinite gradient skipped: 7
Nonfinite gradient skipped: 9
Nonfinite gradient skipped: 12
Step 50/700: teacher match 0.635, PPL ratio 0.885, best 50
Step 100/700: teacher match 0.635, PPL ratio 1.010, best 100
Step 150/700: teacher match 0.635, PPL ratio 0.974, best 150
Step 200/700: teacher match 0.635, PPL ratio 0.927, best 150
Step 250/700: teacher match 0.635, PPL ratio 0.905, best 250
Step 300/700: teacher match 0.583, PPL ratio 1.373, best 250
Step 350/700: teacher match 0.646, PPL ratio 1.024, best 350
Nonfinite gradient skipped: 394
Step 400/700: teacher match 0.646, PPL ratio 0.843, best 400
Step 450/700: teacher match 0.656, PPL ratio 0.883, best 450
Step 500/700: teacher match 0.646, PPL ratio 0.958, best 450
Step 550/700: teacher match 0.

In [7]:
#@title 6. Independent sealed HOLDOUT and honest FP16 latency measurements
# FIRST AND ONLY holdout access; never use it to choose training steps.
holdout_rows=[make_record(s) for s in holdout_text]
teacher.to(device).eval().requires_grad_(False)
holdout_rows=attach_reference(holdout_rows)
HOLDOUT=metrics(student,holdout_rows)
print("SEALED HOLDOUT",HOLDOUT)

@torch.inference_mode()
def median_prefill_ms(model,tokens,repeats=8):
    model.eval()
    x=tokens.to(device)
    for _ in range(2):
        with torch.autocast("cuda",dtype=torch.float16):
            model(input_ids=x,use_cache=False,logits_to_keep=1)
    torch.cuda.synchronize()
    durations=[]
    for _ in range(repeats):
        begin=torch.cuda.Event(enable_timing=True)
        end=torch.cuda.Event(enable_timing=True)
        begin.record()
        with torch.autocast("cuda",dtype=torch.float16):
            model(input_ids=x,use_cache=False,logits_to_keep=1)
        end.record();end.synchronize()
        durations.append(float(begin.elapsed_time(end)))
    return float(np.median(durations))

timings=[]
for length in (64,128,192,256):
    # One authentic holdout excerpt; pad by repetition (not zeros) if needed.
    tokens=holdout_rows[0]["input_ids"].to(device)
    if tokens.shape[1]<length:
        reps=math.ceil(length/tokens.shape[1])
        tokens=tokens.repeat(1,reps)[:,:length]
    else:tokens=tokens[:,:length]
    teacher_ms=median_prefill_ms(teacher,tokens)
    student_ms=median_prefill_ms(student,tokens)
    timings.append({"input_tokens":length,"teacher_ms":teacher_ms,
                    "student_ms":student_ms,
                    "speedup":teacher_ms/max(1e-9,student_ms),
                    "dtype":"float16","batch_size":1,"use_cache":False})
    print("Latency",timings[-1])
teacher.cpu();gc.collect();torch.cuda.empty_cache()


Reference logits 0 / 128
Reference logits 25 / 128
Reference logits 50 / 128
Reference logits 75 / 128
Reference logits 100 / 128
Reference logits 125 / 128
SEALED HOLDOUT {'n': 128, 'top1_agreement': 0.59375, 'mean_js': 0.19820778832945507, 'mean_kl': 1.6146388076012954, 'student_nll': 1.5297504717018455, 'teacher_nll': 1.6310891566390637, 'ppl_ratio': 0.9036269362214624}
Latency {'input_tokens': 64, 'teacher_ms': 60.50366401672363, 'student_ms': 52.39140701293945, 'speedup': 1.1548394568174252, 'dtype': 'float16', 'batch_size': 1, 'use_cache': False}
Latency {'input_tokens': 128, 'teacher_ms': 68.5851821899414, 'student_ms': 53.19427299499512, 'speedup': 1.2893339513521385, 'dtype': 'float16', 'batch_size': 1, 'use_cache': False}
Latency {'input_tokens': 192, 'teacher_ms': 74.9542236328125, 'student_ms': 52.1013126373291, 'speedup': 1.4386244767873648, 'dtype': 'float16', 'batch_size': 1, 'use_cache': False}
Latency {'input_tokens': 256, 'teacher_ms': 80.31334686279297, 'student_ms':

In [8]:
#@title 7. Chat-quality tests (greedy, NO CACHE; text only)
TEST_MESSAGES=[
    "Explain in two sentences why the sky appears blue.",
    "What is 17 multiplied by 23? Answer with the number.",
    "Write one friendly greeting in German.",
    "Explain the difference between recurrent and full attention.",
    "Give two safety considerations when operating a small drone."
]
@torch.inference_mode()
def greedy_text(model,prompt,new_tokens=24):
    messages=[{"role":"user","content":prompt}]
    try:
        try:
            obj=tokenizer.apply_chat_template(
                messages,tokenize=True,add_generation_prompt=True,
                return_tensors="pt",return_dict=True,enable_thinking=False)
        except TypeError:
            obj=tokenizer.apply_chat_template(
                messages,tokenize=True,add_generation_prompt=True,
                return_tensors="pt",return_dict=True)
        if isinstance(obj,torch.Tensor): ins=obj
        elif hasattr(obj,"input_ids"):ins=obj.input_ids
        elif isinstance(obj,dict): ins=obj["input_ids"]
        else:ins=torch.as_tensor(obj,dtype=torch.long)
    except Exception as exc:
        print("Falling back to plaintext prompt:",type(exc).__name__)
        ins=tokenizer("User: "+prompt+"\nAssistant:",
                      return_tensors="pt")["input_ids"]
    if isinstance(ins,list):ins=torch.tensor(ins)
    if ins.ndim==1:ins=ins[None,:]
    assert ins.ndim==2
    ins=ins.to(device);initial=ins.shape[-1]
    for _ in range(new_tokens):
        with torch.autocast("cuda",dtype=torch.float16):
            logits=model(input_ids=ins,use_cache=False,
                         logits_to_keep=1).logits[:,-1]
        next_id=logits.argmax(-1,keepdim=True)
        ins=torch.cat((ins,next_id),-1)
        if int(next_id[0,0])==tokenizer.eos_token_id:break
    return tokenizer.decode(ins[0,initial:],skip_special_tokens=True)
teacher.to(device).eval()
chat_rows=[]
for prompt in TEST_MESSAGES:
    row={"prompt":prompt}
    try:
        row["teacher"]=greedy_text(teacher,prompt)
        row["student"]=greedy_text(student,prompt)
    except Exception as exc:
        row["error"]=type(exc).__name__+": "+str(exc)
        print("Chat test failed (metrics/export will continue):",repr(exc))
    chat_rows.append(row)
    print("\nUSER:",prompt,"\nSUBSTITUTE:",row.get("student","FAILED"))
# A fully collapsed or punctuation-only answer is NOT a pass.
import re
def usable_chat_answer(answer):
    answer=(answer or "").strip()
    words=re.findall(r"\w+",answer,flags=re.UNICODE)
    if not words:return False
    if len(set(words))>=3:return True
    if len(words)==1 and any(ch.isdigit() for ch in answer):return True
    if len(words)>=2 and len(set(words))>=2:return True
    return False
for row in chat_rows:
    row["student_usable"]=usable_chat_answer(row.get("student",""))
CHAT_USABLE=sum(bool(x["student_usable"]) for x in chat_rows)
print("Usable chat answers:",CHAT_USABLE,"/",len(chat_rows))
(REPORTS/"chat_examples.json").write_text(json.dumps(chat_rows,indent=2))
teacher.cpu();gc.collect();torch.cuda.empty_cache()



USER: Explain in two sentences why the sky appears blue. 
SUBSTITUTE: An a a a a a a important and the the the the the the the the the the the the the the the

USER: What is 17 multiplied by 23? Answer with the number. 
SUBSTITUTE: #

#

#

#

#

#

# **

# **

# **

# **



USER: Write one friendly greeting in German. 
SUBSTITUTE: Here of a a a a a important and important and important and important and important topics and important topics that important topics that

USER: Explain the difference between recurrent and full attention. 
SUBSTITUTE: ########### **

## **



### **

### **



USER: Give two safety considerations when operating a small drone. 
SUBSTITUTE: Here of A **

An a a a a **

#

#

# **

#####
Usable chat answers: 3 / 5


In [9]:
#@title 8. Quality gate, FP16 save/reload parity, and local model files
median_speed=float(np.median([r["speedup"] for r in timings]))
qualified=bool(
    HOLDOUT["top1_agreement"]>=MIN_TEACHER_AGREEMENT
    and HOLDOUT["ppl_ratio"]<=MAX_PERPLEXITY_RATIO
    and HOLDOUT["mean_js"]<=MAX_JS
    and median_speed>=MIN_SPEEDUP
    and DATA_SOURCE=="wikitext-2-raw-v1"
    and FINAL_DEV["top1_agreement"]>=0.85
    and FINAL_DEV["mean_js"]<=0.06
    and CHAT_USABLE>=4)
checks={
    "minimum_top1_agreement":MIN_TEACHER_AGREEMENT,
    "maximum_perplexity_ratio":MAX_PERPLEXITY_RATIO,
    "maximum_js_divergence":MAX_JS,
    "minimum_median_speedup":MIN_SPEEDUP,
    "minimum_dev_agreement":0.85,
    "maximum_dev_js":0.06,
    "minimum_usable_chat_responses":4}
manifest={
    "architecture":"Qwen3.5_0.8B_All24_WindowAnchor_V3",
    "v2_previous_run":{"dev_teacher_agreement":0.59375,
        "holdout_teacher_agreement":0.59375,
        "holdout_mean_js":0.2004319814926324,
        "median_prefill_speedup":1.0356892296616755,
        "all_five_chat_tests_repetitive_punctuation":True},
    "source_model":MODEL_ID,"dtype":"float16",
    "original_layers":24,"mixers_replaced":{"delta":18,"full":6},
    "attention_settings":{"delta_window":DELTA_WINDOW,
                          "full_window":FULL_WINDOW,"anchors":ANCHORS,"chunk_size":CHUNK},
    "no_cenn":True,"no_delta_recurrence":True,"no_kv_cache":True,
    "text_only":True,"ffn_norm_sha256":TEACHER_FFN_SHA,
    "training_steps_budget":TRAIN_STEPS,"actual_training_steps":actual_train_steps,
    "selected_step":best_step,"early_stop_patience":EVAL_PATIENCE,
    "layerwise_alignment":LOCAL_HISTORY,
    "chat_training_prompts":len(CHAT_TRAIN_PROMPTS),
    "chat_continuation_records":len(chat_train_records),
    "chat_usable_count":CHAT_USABLE,
    "short_attention_dense_fused":True,
    "untuned_dev":BASELINE_DEV,"post_local_dev":LOCAL_DEV,
    "dataset":DATA_SOURCE,"dev":FINAL_DEV,
    "holdout":HOLDOUT,"benchmark":timings,
    "median_speedup":median_speed,
    "publication_gate":checks,"qualified":qualified,
    "utc":datetime.now(timezone.utc).isoformat()
}
(REPORTS/"final_report.json").write_text(json.dumps(manifest,indent=2))
print("QUALITY GATE:",qualified, "median speedup",median_speed)
print("Full model local export (even if quality gate fails)...")
reload_ok=False
parity=None
reload_error=None
try:
    wa.save_custom(student,tokenizer,ARTIFACTS,manifest)
    assert (ARTIFACTS/"model.safetensors").exists()
    before=student.to(device).eval()
    probe=dev_rows[0]["input_ids"].to(device)
    with torch.inference_mode(),torch.autocast("cuda",dtype=torch.float16):
        original_logits=before(input_ids=probe,use_cache=False,logits_to_keep=1).logits
    student.cpu()
    gc.collect();torch.cuda.empty_cache()
    reloaded,loaded_tokenizer,loaded_counts=wa.load_custom(ARTIFACTS,device=device)
    assert loaded_counts=={"delta_replaced":18,"full_replaced":6}
    assert ff_norm_fingerprint(reloaded)==TEACHER_FFN_SHA
    with torch.inference_mode(),torch.autocast("cuda",dtype=torch.float16):
        roundtrip_logits=reloaded(input_ids=probe,use_cache=False,logits_to_keep=1).logits
    parity=float((original_logits.float()-roundtrip_logits.float()).abs().max())
    assert parity<0.01, ("Model reload logit mismatch",parity)
    reload_ok=True
    del reloaded
    gc.collect();torch.cuda.empty_cache()
    print("STRICT 24-LAYER FP16 RELOAD PASSED, max difference:",parity)
except Exception as exc:
    # Never publish a model if its exported weights cannot be reloaded!
    qualified=False
    reload_error=repr(exc)
    print("EXPORT/RELOAD FAILED. HF upload disabled, but statistics will download:",repr(exc))
finally:
    manifest["roundtrip_max_logit_abs"]=parity
    manifest["reload_verified"]=reload_ok
    manifest["reload_error"]=reload_error
    manifest["qualified"]=bool(qualified and reload_ok)
    qualified=manifest["qualified"]
    (REPORTS/"final_report.json").write_text(json.dumps(manifest,indent=2))
    if (ARTIFACTS/"window_anchor_manifest.json").exists():
        (ARTIFACTS/"window_anchor_manifest.json").write_text(json.dumps(manifest,indent=2))


QUALITY GATE: False median speedup 1.3639792140697518
Full model local export (even if quality gate fails)...
EXPORT/RELOAD FAILED. HF upload disabled, but statistics will download: AssertionError(('Model reload logit mismatch', 0.015625))


In [10]:
#@title 9. Full model documentation, reproducibility and final statistics
card=f"""---
license: apache-2.0
library_name: pytorch
tags:
- qwen3.5
- window-anchor
- attention-substitution
- no-cenn
- research
---

# Qwen3.5-0.8B Window+Anchor V3 Optimized Attention (NO CeNN)

Text-only research checkpoint derived from {MODEL_ID} (Apache-2.0).

**18 DeltaNet and 6 full attention mixers replaced; FFNs/norms remain
unchanged.** Gated DeltaNet RECURRENCE is removed. Window/anchor attention
is **causal**. This implementation DOES NOT support KV-cache decoding,
images or video, and is NOT plug-and-play via AutoModel.from_pretrained.

Training examples: {TRAIN_EXAMPLES} | DEV: {DEV_EXAMPLES} | sealed HOLDOUT: {HOLDOUT_EXAMPLES}.
Best step: {best_step}.
Holdout top-1 teacher agreement: {HOLDOUT['top1_agreement']:.3%}.
Holdout NLL-based perplexity ratio: {HOLDOUT['ppl_ratio']:.3f}.
Median end-to-end FP16 prefill speedup: {median_speed:.3f}x.
Publication gate passed: {qualified}. Chat responses usable: {CHAT_USABLE}/{len(chat_rows)}. Export reload verified: {reload_ok}.

## Load locally or from Hub

Install recent transformers, torch, and safetensors, then import
\`tinycenn_lm.qwen35_window_anchor_v3.load_custom\` from the
[TinyCeNN-LM source](https://github.com/vtavakkoli/TinyCeNN-LM).

\`\`\`python
from tinycenn_lm.qwen35_window_anchor_v3 import load_custom
model, tokenizer, counts = load_custom("PATH_TO_DOWNLOADED_REPOSITORY",device="cuda")
inputs = tokenizer("Hello",return_tensors="pt").to("cuda")
out = model(**inputs,use_cache=False,logits_to_keep=1)
print(tokenizer.decode(out.logits[:,-1].argmax(-1)))
\`\`\`

For Hub download use huggingface_hub.snapshot_download and pass the
resulting local directory to load_custom. For chat, use full-sequence
greedy generation with \`use_cache=False\`. This model does NOT implement
native recurrent DeltaNet caches or incremental KV generation.
"""
(ARTIFACTS/"README.md").write_text(card)
(ARTIFACTS/"requirements.txt").write_text(
    "torch>=2.4\nsafetensors>=0.5\nhuggingface_hub>=0.34\n"
    "git+https://github.com/huggingface/transformers.git@main\n"
    "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@66a244e8e8ee5c8b9b44f15a888d75e59761e561\n")
(REPORTS/"summary.json").write_text(json.dumps({
    "qualified":qualified,"best_step":best_step,"final_dev":FINAL_DEV,
    "sealed_holdout":HOLDOUT,"timings":timings,"checks":checks,
    "fp16_reload_max_abs_error":parity},indent=2))


<>:52: SyntaxWarning: invalid escape sequence '\`'
<>:52: SyntaxWarning: invalid escape sequence '\`'
/tmp/ipykernel_2168/2461373581.py:52: SyntaxWarning: invalid escape sequence '\`'
  "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@66a244e8e8ee5c8b9b44f15a888d75e59761e561\n")


1789

In [11]:
#@title 10. Hugging Face upload ONLY IF the model passed ALL quality checks
HF_STATUS={"qualified":qualified,"attempted":False,"uploaded":False,
           "reason":"Quality gate failed; local model and ZIPs were saved."}
if qualified and UPLOAD_IF_QUALIFIED:
    try:
        token=get_token()
        if not token:
            login()
            token=get_token()
        assert token,"Hugging Face write token needed"
        api=HfApi(token=token)
        username=api.whoami()["name"]
        repo_id=f"{username}/{HF_NAME}"
        HF_STATUS["attempted"]=True
        api.create_repo(repo_id,repo_type="model",private=HF_PRIVATE,exist_ok=True)
        api.upload_folder(
            repo_id=repo_id,repo_type="model",folder_path=str(ARTIFACTS),
            commit_message="Causal all-24 window-anchor substitution verified on sealed holdout")
        files_on_hf=set(api.list_repo_files(repo_id,repo_type="model"))
        required={"model.safetensors","window_anchor_manifest.json",
                  "qwen35_window_anchor_v3.py","config.json","README.md"}
        assert required<=files_on_hf,(required-files_on_hf)
        HF_STATUS.update({"uploaded":True,"repo_id":repo_id,
                          "url":f"https://huggingface.co/{repo_id}","reason":"Quality gate passed"})
    except Exception as exc:
        HF_STATUS["reason"]=repr(exc)
        print("HF upload error; local model still saved:",repr(exc))
elif not UPLOAD_IF_QUALIFIED:
    HF_STATUS["reason"]="Upload disabled by user."
(REPORTS/"hf_upload_status.json").write_text(json.dumps(HF_STATUS,indent=2))
print("HUGGING FACE STATUS",HF_STATUS)
print("HF status saved. Final ZIP cell will include the upload-status report.")


HUGGING FACE STATUS {'qualified': False, 'attempted': False, 'uploaded': False, 'reason': 'Quality gate failed; local model and ZIPs were saved.'}
HF status saved. Final ZIP cell will include the upload-status report.


In [12]:
#@title 11. ZIP final statistics INCLUDING upload status and download
from IPython.display import display, FileLink
STATS_ZIP=BASE/"Qwen35_0p8B_All24_V3_Final_Statistics.zip"
MODEL_ZIP=BASE/"Qwen35_0p8B_All24_V3_Standalone_Model.zip"
report_paths=sorted(p for p in REPORTS.rglob("*") if p.is_file())
assert (REPORTS/"final_report.json") in report_paths
with zipfile.ZipFile(STATS_ZIP,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in report_paths:
        z.write(p,"statistics/"+p.relative_to(REPORTS).as_posix())
if (ARTIFACTS/"model.safetensors").is_file():
    with zipfile.ZipFile(MODEL_ZIP,"w",allowZip64=True) as z:
        for p in sorted(p for p in ARTIFACTS.rglob("*") if p.is_file()):
            compression=(zipfile.ZIP_STORED if p.suffix==".safetensors"
                         else zipfile.ZIP_DEFLATED)
            z.write(p,"model/"+p.relative_to(ARTIFACTS).as_posix(),
                    compress_type=compression)
        for p in report_paths:
            z.write(p,"statistics/"+p.relative_to(REPORTS).as_posix(),
                    compress_type=zipfile.ZIP_DEFLATED)
else:
    print("No model.safetensors was exported; statistics ZIP will still download.")
for p in (STATS_ZIP,MODEL_ZIP):
    if not p.exists():continue
    with zipfile.ZipFile(p) as z:
        assert z.testzip() is None
    print("ZIP verified:",p,"MiB",round(p.stat().st_size/2**20,2))
display(FileLink(str(STATS_ZIP)))
if MODEL_ZIP.exists():display(FileLink(str(MODEL_ZIP)))
try:
    from google.colab import files
    if AUTO_DOWNLOAD_STATS:files.download(str(STATS_ZIP))
    if AUTO_DOWNLOAD_MODEL_ZIP and MODEL_ZIP.exists():files.download(str(MODEL_ZIP))
except Exception as exc:
    print("Browser auto-download failed:",repr(exc))
    print("Use the displayed FileLinks or Colab Files sidebar.")


ZIP verified: /content/Qwen35_0p8B_All24_WindowAnchor_V3/Qwen35_0p8B_All24_V3_Final_Statistics.zip MiB 0.01
ZIP verified: /content/Qwen35_0p8B_All24_WindowAnchor_V3/Qwen35_0p8B_All24_V3_Standalone_Model.zip MiB 1438.37


/content/Qwen35_0p8B_All24_WindowAnchor_V3/Qwen35_0p8B_All24_V3_Final_Statistics.zip

/content/Qwen35_0p8B_All24_WindowAnchor_V3/Qwen35_0p8B_All24_V3_Standalone_Model.zip

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>